<a href="https://colab.research.google.com/github/priyanka1994-as/git-basics-demo/blob/main/task3_analysis_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
1 — Make the API Calls

In [ ]:
import requests
import time

# URLs
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

headers = {"User-Agent": "TrendPulse/1.0"}

# Categories with keywords
categories = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Function to match category
def get_category(title):
    title = title.lower()
    for category, keywords in categories.items():
        for keyword in keywords:
            if keyword in title:
                return category
    return None


# Step 1: Fetch top story IDs
try:
    response = requests.get(TOP_STORIES_URL, headers=headers)
    response.raise_for_status()
    story_ids = response.json()[:500]
except Exception as e:
    print("Failed to fetch top stories:", e)
    story_ids = []

print(f"Fetched {len(story_ids)} story IDs")

# Step 2: Fetch stories category-wise
stories = []

for category in categories.keys():
    print(f"\nProcessing category: {category}")

    for story_id in story_ids:
        try:
            res = requests.get(ITEM_URL.format(story_id), headers=headers)
            res.raise_for_status()
            story = res.json()

            # Skip invalid stories
            if story is None or "title" not in story:
                continue

            # Check category match
            if get_category(story["title"]) == category:
                stories.append({
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0)
                })

        except Exception as e:
            print(f"Error fetching story {story_id}: {e}")
            continue

    # ✅ Sleep AFTER each category loop
    time.sleep(2)

print(f"\nTotal collected stories: {len(stories)}")

Fetched 500 story IDs

Processing category: technology

Processing category: worldnews

Processing category: sports

Processing category: science

Processing category: entertainment

Total collected stories: 218


In [ ]:
2 — Extract the Fields

In [ ]:
import requests
import time
from datetime import datetime

# URLs
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

headers = {"User-Agent": "TrendPulse/1.0"}

# Categories
categories = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Function to assign category
def get_category(title):
    title = title.lower()
    for category, keywords in categories.items():
        for keyword in keywords:
            if keyword in title:
                return category
    return None


# Fetch top IDs
response = requests.get(TOP_STORIES_URL, headers=headers)
story_ids = response.json()[:500]

stories = []

# Process category-wise
for category in categories.keys():
    print(f"\nProcessing {category}...")
    count = 0

    for story_id in story_ids:
        if count >= 25:  # ✅ limit per category
            break

        try:
            res = requests.get(ITEM_URL.format(story_id), headers=headers)
            story = res.json()

            if story is None or "title" not in story:
                continue

            # Check category match
            if get_category(story["title"]) == category:

                stories.append({
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),  # ✅ new field
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # ✅ new field
                })

                count += 1

        except Exception as e:
            print(f"Error fetching {story_id}: {e}")
            continue

    print(f"Collected {count} stories for {category}")

    # ✅ sleep per category (important for marks)
    time.sleep(2)

print(f"\nTotal collected stories: {len(stories)}")


Processing technology...
Collected 25 stories for technology

Processing worldnews...
Collected 11 stories for worldnews

Processing sports...
Collected 8 stories for sports

Processing science...
Collected 11 stories for science

Processing entertainment...
Collected 25 stories for entertainment

Total collected stories: 80


In [ ]:
3 — Save to a JSON File

In [6]:
import os
import json
from datetime import datetime

# Step 1: Create folder if it doesn't exist
folder_name = "data"
os.makedirs(folder_name, exist_ok=True)

# Step 2: Create filename with date
date_str = datetime.now().strftime("%Y%m%d")
file_path = f"{folder_name}/trends_{date_str}.json"

# Step 3: Save to JSON file
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(stories, f, indent=4)

# Step 4: Print total count
print(f"\nSaved {len(stories)} stories to {file_path}")


Saved 80 stories to data/trends_20260406.json


In [ ]:
1 — Load the JSON File

In [7]:
import pandas as pd

# Load JSON file
file_path = "data/trends_20260406.json"  # update filename if needed
df = pd.read_json(file_path)

# Print number of rows
print(f"Total rows loaded: {len(df)}")

Total rows loaded: 80


In [ ]:
2 — Clean the Data

In [8]:
import pandas as pd

# Load file
df = pd.read_json("data/trends_20260406.json")

print("Before Cleaning:", df.shape)

# 1️⃣ Remove duplicates (based on post_id)
df = df.drop_duplicates(subset="post_id")

# 2️⃣ Handle missing values (drop important missing fields)
df = df.dropna(subset=["post_id", "title", "score"])

# 3️⃣ Fix data types
df["score"] = df["score"].astype(int)
df["num_comments"] = df["num_comments"].astype(int)

# 4️⃣ Remove low-quality stories (score < 5)
df = df[df["score"] >= 5]

# 5️⃣ Remove extra whitespace in title
df["title"] = df["title"].str.strip()

# Final result
print("After Cleaning:", df.shape)
print(f"Rows remaining after cleaning: {len(df)}")

Before Cleaning: (80, 7)
After Cleaning: (77, 7)
Rows remaining after cleaning: 77


In [11]:
import os

# Create 'data' directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save the cleaned DataFrame to a CSV file
df.to_csv("data/trends_clean.csv", index=False)
print("Saved cleaned data to data/trends_clean.csv")

Saved cleaned data to data/trends_clean.csv


In [ ]:
1 — Load and Explore

In [12]:
import pandas as pd
import numpy as np
import os

# Step 1: Load cleaned CSV
df = pd.read_csv("data/trends_clean.csv")

# Step 2: Basic Exploration
print("First 5 rows:\n", df.head())
print("\nShape:", df.shape)

print("\nAverage Score:", df["score"].mean())
print("Average Comments:", df["num_comments"].mean())

# Step 3: Find Patterns (group by category)
print("\nAverage Score per Category:")
print(df.groupby("category")["score"].mean())

print("\nTotal Stories per Category:")
print(df["category"].value_counts())

# Step 4: Add New Columns

# 1️⃣ Engagement Score (score + comments)
df["engagement"] = df["score"] + df["num_comments"]

# 2️⃣ Popularity Label using NumPy
df["popularity"] = np.where(df["score"] > df["score"].mean(), "High", "Low")

# Step 5: Save new CSV for Task 4
os.makedirs("data", exist_ok=True)
output_path = "data/trends_enriched.csv"

df.to_csv(output_path, index=False)

print(f"\nSaved enriched data to {output_path}")

First 5 rows:
     post_id                                              title    category  \
0  47657699   SideX – A Tauri-based port of Visual Studio Code  technology   
1  47655408  Show HN: I built a tiny LLM to demystify how l...  technology   
2  47652007  Show HN: Real-time AI (audio/video in, voice o...  technology   
3  47655367  Show HN: Gemma Gem – AI model embedded in a br...  technology   
4  47651540  Running Gemma 4 locally with LM Studio's new h...  technology   

   score  num_comments      author         collected_at  
0     32            15      0x1997  2026-04-06 07:24:40  
1    339            31  armanified  2026-04-06 07:24:41  
2     45             2      karimf  2026-04-06 07:24:44  
3     57            10    ikessler  2026-04-06 07:24:44  
4    248            59   vbtechguy  2026-04-06 07:24:45  

Shape: (77, 7)

Average Score: 180.6883116883117
Average Comments: 81.63636363636364

Average Score per Category:
category
entertainment    171.666667
science         

In [13]:
import pandas as pd

# Step 1: Load cleaned CSV
df = pd.read_csv("data/trends_clean.csv")

# Step 2: Print first 5 rows
print("First 5 rows:")
print(df.head())

# Step 3: Print shape
print("\nShape of DataFrame:")
print(df.shape)

# Step 4: Compute averages
avg_score = df["score"].mean()
avg_comments = df["num_comments"].mean()

print(f"\nAverage Score: {avg_score:.2f}")
print(f"Average Comments: {avg_comments:.2f}")

First 5 rows:
    post_id                                              title    category  \
0  47657699   SideX – A Tauri-based port of Visual Studio Code  technology   
1  47655408  Show HN: I built a tiny LLM to demystify how l...  technology   
2  47652007  Show HN: Real-time AI (audio/video in, voice o...  technology   
3  47655367  Show HN: Gemma Gem – AI model embedded in a br...  technology   
4  47651540  Running Gemma 4 locally with LM Studio's new h...  technology   

   score  num_comments      author         collected_at  
0     32            15      0x1997  2026-04-06 07:24:40  
1    339            31  armanified  2026-04-06 07:24:41  
2     45             2      karimf  2026-04-06 07:24:44  
3     57            10    ikessler  2026-04-06 07:24:44  
4    248            59   vbtechguy  2026-04-06 07:24:45  

Shape of DataFrame:
(77, 7)

Average Score: 180.69
Average Comments: 81.64


In [ ]:
2 — Basic Analysis with NumPy

In [14]:
import pandas as pd
import numpy as np

# Load cleaned CSV
df = pd.read_csv("data/trends_clean.csv")

# Convert columns to NumPy arrays
scores = df["score"].to_numpy()
comments = df["num_comments"].to_numpy()

# 1️⃣ Mean, Median, Standard Deviation of score
print("Score Statistics:")
print("Mean:", np.mean(scores))
print("Median:", np.median(scores))
print("Standard Deviation:", np.std(scores))

# 2️⃣ Highest and Lowest score
print("\nScore Range:")
print("Highest Score:", np.max(scores))
print("Lowest Score:", np.min(scores))

# 3️⃣ Category with most stories
category_counts = df["category"].value_counts()
top_category = category_counts.idxmax()

print("\nCategory with most stories:", top_category)

# 4️⃣ Story with most comments
max_comments_index = np.argmax(comments)
top_story = df.iloc[max_comments_index]

print("\nMost Commented Story:")
print("Title:", top_story["title"])
print("Comments:", top_story["num_comments"])

Score Statistics:
Mean: 180.6883116883117
Median: 70.0
Standard Deviation: 250.4461089730623

Score Range:
Highest Score: 1076
Lowest Score: 5

Category with most stories: technology

Most Commented Story:
Title: Tell HN: Anthropic no longer allowing Claude Code subscriptions to use OpenClaw
Comments: 816


In [ ]:
3 — Add New Columns

In [16]:
import pandas as pd

# Load cleaned data
df = pd.read_csv("data/trends_clean.csv")

# 1️⃣ Engagement column
df["engagement"] = df["num_comments"] / (df["score"] + 1)

# 2️⃣ is_popular column
average_score = df["score"].mean()
df["is_popular"] = df["score"] > average_score

# Display result
print(df.head())

    post_id                                              title    category  \
0  47657699   SideX – A Tauri-based port of Visual Studio Code  technology   
1  47655408  Show HN: I built a tiny LLM to demystify how l...  technology   
2  47652007  Show HN: Real-time AI (audio/video in, voice o...  technology   
3  47655367  Show HN: Gemma Gem – AI model embedded in a br...  technology   
4  47651540  Running Gemma 4 locally with LM Studio's new h...  technology   

   score  num_comments      author         collected_at  engagement  \
0     32            15      0x1997  2026-04-06 07:24:40    0.454545   
1    339            31  armanified  2026-04-06 07:24:41    0.091176   
2     45             2      karimf  2026-04-06 07:24:44    0.043478   
3     57            10    ikessler  2026-04-06 07:24:44    0.172414   
4    248            59   vbtechguy  2026-04-06 07:24:45    0.236948   

   is_popular  
0       False  
1        True  
2       False  
3       False  
4        True  


In [ ]:
4 — Save the Result

In [15]:
import os

# Create folder if not exists
os.makedirs("data", exist_ok=True)

# Save DataFrame to CSV
output_path = "data/trends_analysed.csv"
df.to_csv(output_path, index=False)

# Confirmation message
print(f"Data successfully saved to {output_path}")

Data successfully saved to data/trends_analysed.csv
